# Notebook 4 — Build the ChromaDB Vector Store

This notebook reads the text documents created by **Notebook 3**, splits them into smaller chunks, creates local embeddings, and stores the vectors in ChromaDB on Google Drive.

**Project location:** `/content/drive/MyDrive/RAG UFC-AI-Analyst`

**Important**
- Run the cells from top to bottom.
- Use the same Google account/Drive where Notebook 2 and Notebook 3 saved the project.
- The notebook uses batches of 500 chunks to avoid ChromaDB's maximum-batch-size error.
- It only deletes the `ufc_rag` collection in this notebook's target ChromaDB database before rebuilding it. It does not delete your CSV, DuckDB database, or generated documents.


In [1]:
# Cell 1 — Install required packages
%pip install -q chromadb sentence-transformers


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [2]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# Cell 3 — Set project paths and verify Notebook 3 output
from pathlib import Path
import json
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/RAG UFC-AI-Analyst")
DOCUMENTS_DIR = PROJECT_ROOT / "documents"
MANIFEST_PATH = DOCUMENTS_DIR / "manifest.json"
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore" / "chroma_db"
COLLECTION_NAME = "ufc_rag"

print("Project exists:", PROJECT_ROOT.is_dir(), PROJECT_ROOT)
print("Documents directory exists:", DOCUMENTS_DIR.is_dir(), DOCUMENTS_DIR)
print("Manifest exists:", MANIFEST_PATH.is_file(), MANIFEST_PATH)
print("Vector store path:", VECTORSTORE_DIR)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}. "
        "Check the folder name in MyDrive."
    )

if not DOCUMENTS_DIR.is_dir():
    raise FileNotFoundError(
        f"Documents folder not found: {DOCUMENTS_DIR}. Run Notebook 3 first."
    )

document_files = sorted(
    p for p in DOCUMENTS_DIR.rglob("*.txt")
    if p.is_file()
)
print("Text documents found:", len(document_files))

if not document_files:
    raise FileNotFoundError(
        "No .txt documents were found under documents/. "
        "Check that Notebook 3 saved its output to this Google Drive project."
    )

if MANIFEST_PATH.is_file():
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    print("Manifest loaded successfully. Top-level type:", type(manifest).__name__)
else:
    print("Warning: manifest.json was not found; continuing using the .txt files.")


Project exists: True /content/drive/MyDrive/RAG UFC-AI-Analyst
Documents directory exists: True /content/drive/MyDrive/RAG UFC-AI-Analyst/documents
Manifest exists: False /content/drive/MyDrive/RAG UFC-AI-Analyst/documents/manifest.json
Vector store path: /content/drive/MyDrive/RAG UFC-AI-Analyst/vectorstore/chroma_db
Text documents found: 5044


In [4]:
# Cell 4 — Install/import dependencies and configure embedding model
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Small, general-purpose sentence embedding model; downloaded on first use.
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_function = SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME
)

print("ChromaDB version:", chromadb.__version__)
print("Embedding model:", EMBEDDING_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB version: 1.5.9
Embedding model: all-MiniLM-L6-v2


## Why chunk documents?

A whole fighter or event file may be too large or contain several facts. Chunking creates smaller overlapping passages so retrieval can return the most relevant section rather than the entire file. Metadata records the source file and document category so later retrieval results can be traced back to their source.


In [5]:
# Cell 5 — Read text files and split them into chunks
# Keep this implementation simple and deterministic; no LangChain dependency is needed.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

def split_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = text.strip()
    if not text:
        return []

    # Prefer splitting on paragraph/line boundaries, then fall back to fixed windows.
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = min(start + chunk_size, text_length)

        if end < text_length:
            # Try to end at a nearby newline or space to avoid cutting words.
            newline_pos = text.rfind("\\n", start, end)
            space_pos = text.rfind(" ", start, end)
            preferred = max(newline_pos, space_pos)
            if preferred > start + int(chunk_size * 0.65):
                end = preferred

        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)

        if end >= text_length:
            break

        next_start = max(end - overlap, start + 1)
        start = next_start

    return chunks

documents = []
metadatas = []
ids = []

for file_path in document_files:
    text = file_path.read_text(encoding="utf-8", errors="replace").strip()
    if not text:
        continue

    relative_path = file_path.relative_to(DOCUMENTS_DIR).as_posix()
    category = file_path.parent.name
    pieces = split_text(text)

    for chunk_index, piece in enumerate(pieces):
        documents.append(piece)
        metadatas.append({
            "source": relative_path,
            "category": category,
            "chunk_index": int(chunk_index),
            "file_name": file_path.name,
        })
        ids.append(f"{relative_path}::chunk_{chunk_index}")

print("Source text files:", len(document_files))
print("Total chunks:", len(documents))
print("Example metadata:", metadatas[0] if metadatas else "No chunks")
print("Example chunk:\\n", documents[0][:500] if documents else "No chunks")

if not documents:
    raise ValueError("No non-empty chunks were created. Check the source text files.")


Source text files: 5044
Total chunks: 11472
Example metadata: {'source': 'fighters/AALON_CRUZ_003d82fa384ca1d0.txt', 'category': 'fighters', 'chunk_index': 0, 'file_name': 'AALON_CRUZ_003d82fa384ca1d0.txt'}
Example chunk:\n # UFC Fighter Profile: AALON CRUZ
Fighter ID: 003d82fa384ca1d0
Date of birth: 1989-09-20 00:00:00
Height: 72.0
Reach: 78.0

## Career summary
Recorded fights in this dataset: 2
Wins based on derived source labels: 0
Losses based on derived source labels: 2
Fights with unknown result: 0

## Recent recorded fights
- Fight ID 0bcb04163f8d8ead: loss vs SPIKE CARLYLE; event: UFC Fight Night: Benavidez vs. Figueiredo; date: 2020-02-29 00:00:00; weight class: Featherweight.
- Fight ID e2abba69c3371618:


In [6]:
# Cell 6 — Create/reopen the persistent ChromaDB database
# Ensure only one notebook/runtime is writing to this database at a time.
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(VECTORSTORE_DIR))

existing_collections = [c.name for c in client.list_collections()]
print("Existing collections:", existing_collections)

# Rebuild this collection so rerunning the notebook does not duplicate old chunks.
if COLLECTION_NAME in existing_collections:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted old '{COLLECTION_NAME}' collection for a clean rebuild.")

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
    metadata={"description": "UFC RAG documents generated by Notebook 3"}
)

print("Created collection:", collection.name)


Existing collections: []
Created collection: ufc_rag


In [7]:
# Cell 7 — Embed and save chunks in small batches
from tqdm.auto import tqdm
import time

BATCH_SIZE = 500  # Safely below ChromaDB's common maximum batch size.
total_chunks = len(documents)

for start in tqdm(range(0, total_chunks, BATCH_SIZE), desc="Embedding batches"):
    end = min(start + BATCH_SIZE, total_chunks)

    collection.add(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
    )

    if (start // BATCH_SIZE) % 5 == 0 or end == total_chunks:
        print(f"Stored {end:,} / {total_chunks:,} chunks")

print("Vector indexing finished.")
print("Collection count:", collection.count())
print("Persistent directory:", VECTORSTORE_DIR)


Embedding batches:   0%|          | 0/23 [00:00<?, ?it/s]

Stored 500 / 11,472 chunks
Stored 3,000 / 11,472 chunks
Stored 5,500 / 11,472 chunks
Stored 8,000 / 11,472 chunks
Stored 10,500 / 11,472 chunks
Stored 11,472 / 11,472 chunks
Vector indexing finished.
Collection count: 11472
Persistent directory: /content/drive/MyDrive/RAG UFC-AI-Analyst/vectorstore/chroma_db


In [8]:
# Cell 8 — Verify stored records and metadata
stored_count = collection.count()
sample = collection.get(limit=3, include=["documents", "metadatas"])

print("Expected chunks:", len(documents))
print("Stored chunks:", stored_count)
print("Counts match:", stored_count == len(documents))

for i, doc_id in enumerate(sample.get("ids", [])):
    print(f"\\n--- Sample {i + 1} ---")
    print("ID:", doc_id)
    print("Metadata:", sample["metadatas"][i])
    print("Text:", sample["documents"][i][:350])

if stored_count != len(documents):
    raise RuntimeError(
        f"Stored count ({stored_count}) does not match expected count ({len(documents)})."
    )


Expected chunks: 11472
Stored chunks: 11472
Counts match: True
\n--- Sample 1 ---
ID: fighters/AALON_CRUZ_003d82fa384ca1d0.txt::chunk_0
Metadata: {'source': 'fighters/AALON_CRUZ_003d82fa384ca1d0.txt', 'category': 'fighters', 'chunk_index': 0, 'file_name': 'AALON_CRUZ_003d82fa384ca1d0.txt'}
Text: # UFC Fighter Profile: AALON CRUZ
Fighter ID: 003d82fa384ca1d0
Date of birth: 1989-09-20 00:00:00
Height: 72.0
Reach: 78.0

## Career summary
Recorded fights in this dataset: 2
Wins based on derived source labels: 0
Losses based on derived source labels: 2
Fights with unknown result: 0

## Recent recorded fights
- Fight ID 0bcb04163f8d8ead: loss vs
\n--- Sample 2 ---
ID: fighters/AARON_BRINK_cf1a88371c8cb690.txt::chunk_0
Metadata: {'file_name': 'AARON_BRINK_cf1a88371c8cb690.txt', 'source': 'fighters/AARON_BRINK_cf1a88371c8cb690.txt', 'category': 'fighters', 'chunk_index': 0}
Text: # UFC Fighter Profile: AARON BRINK
Fighter ID: cf1a88371c8cb690
Date of birth: 1974-11-12 00:00:00
Height: 75.0
Rea

In [9]:
# Cell 9 — Test semantic search
# This confirms that ChromaDB can embed a query and retrieve relevant UFC documents.
query_text = "Which fighter won by submission, and what fight details are available?"

results = collection.query(
    query_texts=[query_text],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

print("Query:", query_text)
for rank, (doc, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ),
    start=1,
):
    print(f"\\nResult {rank} | distance={distance:.4f}")
    print("Source:", metadata.get("source"))
    print("Category:", metadata.get("category"))
    print(doc[:600])


Query: Which fighter won by submission, and what fight details are available?
\nResult 1 | distance=0.3252
Source: fighters/Javad_Mahjoub_e9f9add1481fc11c.txt
Category: fighters
# UFC Fighter Profile: Javad Mahjoub
Fighter ID: e9f9add1481fc11c
Date of birth: 1991-05-26 00:00:00
Height: 73.0
Reach: 76.0

## Career summary
Recorded fights in this dataset: 1
Wins based on derived source labels: 0
Losses based on derived source labels: 1
Fights with unknown result: 0

## Recent recorded fights
- Fight ID 88247eaef34c209b: loss vs Joel Faglier; event: UFC Fight Night: Buckley vs. Malott; date: 2026-10-17 00:00:00; weight class: Heavyweight.
\nResult 2 | distance=0.3322
Source: fighters/ZYGIMANTAS_RAMASKA_641ca2c2ed91b93c.txt
Category: fighters
# UFC Fighter Profile: ZYGIMANTAS RAMASKA
Fighter ID: 641ca2c2ed91b93c
Date of birth: 1996-12-27 00:00:00
Height: 71.0
Reach: 71.0

## Career summary
Recorded fights in this dataset: 1
Wins based on derived source labels: 0
Losses based on derived sou

In [10]:
# Cell 10 — Final verification and close
final_count = collection.count()
print("Notebook 4 completed.")
print("Project:", PROJECT_ROOT)
print("Vector store:", VECTORSTORE_DIR)
print("Collection:", COLLECTION_NAME)
print("Stored chunks:", final_count)
print("Semantic search test: completed")

# PersistentClient writes to disk automatically; no explicit close is required.


Notebook 4 completed.
Project: /content/drive/MyDrive/RAG UFC-AI-Analyst
Vector store: /content/drive/MyDrive/RAG UFC-AI-Analyst/vectorstore/chroma_db
Collection: ufc_rag
Stored chunks: 11472
Semantic search test: completed
